# nuget-hebung-kit — Walkthrough (GitHub Copilot CLI)

Make **GitHub Copilot CLI** drive a large, persisted, subagent-driven **NuGet upgrade ("Hebung")** in a target .NET repo.

**How to read this notebook:** the `powershell`/`bash` cells are real terminal commands. The `/nuget-hebung`, `/skills`, `/handoff` steps are **typed into the Copilot CLI prompt** — a notebook can't drive that, so they're shown in markdown. Run them inside `copilot`.

## 1. Prerequisites
- **GitHub Copilot CLI** installed and logged in (`copilot`, then `/login`).
- **.NET SDK** (the upgrade target).

In [ ]:
copilot --version   # Copilot CLI present?
dotnet --version    # .NET SDK present?

## 2. Get this kit and validate it
Clone the kit next to your target repo, then smoke-check it.

In [ ]:
git clone <your-fork>/nuget-hebung-kit
powershell -NoProfile -ExecutionPolicy Bypass -File nuget-hebung-kit/scripts/smoke-check.ps1
# expect: SMOKE CHECK PASSED

## 3. Make your target repo CLI-ready (automatic)
From inside the kit, point the bootstrap script at your repo. It copies the skills, the model-pinned agents, the context-guard hook, and the risk KB; ensures `AGENTS.md` exists; creates `docs/` persistence; and gitignores `tasks/`.

In [ ]:
cd nuget-hebung-kit
./scripts/bootstrap.ps1 -TargetRepo C:\path\to\your-repo
# (manual alternative + what each file does: see SETUP.md, Option B)

In [ ]:
cd C:\path\to\your-repo
git checkout -b feature/nuget-hebung
git add -A; git commit -m "chore: install NuGet Hebung CLI kit"

## 4. Verify the CLI sees everything
Open the repo in the CLI and inspect what's loaded:

```bash
cd C:\path\to\your-repo
copilot
```

Then inside the CLI:

```text
/env       # instructions, skills, agents, hooks loaded
/skills    # expect: nuget-hebung, handoff
/agent     # expect: nuget-project-investigator, nuget-package-updater
```

## 5. Run the upgrade
In the CLI, in the target repo:

```text
/nuget-hebung
```

Phases: **0** preflight + NuGet feed check → **1** brainstorm scope (TFM bumps? renames? exceptions?) → **2** parallel per-project investigation (claude-opus-4.8 subagents → `docs/nuget-hebung/agentresults/`) → **3** consolidate (dependency + state graphs) → **4** resolve conflicts → **5** approve the ordered plan → **6** parallel execution → **7** full-solution verify. State persists in `docs/nuget-hebung/plan.md`.

## 6. What the investigation runs under the hood (target repo)
The investigator subagents use commands like these per project:

In [ ]:
dotnet list package --outdated
dotnet list package --include-transitive
dotnet list package --vulnerable --include-transitive
dotnet nuget why src/<Project>/<Project>.csproj <PackageId>

## 7. Long sessions & resuming
The `agentStop` context-guard hook forces a handoff when context gets tight. To resume in a fresh CLI session, type:

```text
Read docs/nuget-hebung/plan.md and continue from Resume.
```

You can also persist anytime with `/handoff`.